# Task 1 — QLoRA adapter (Qwen2.5-Coder-7B) trên Kaggle

Chạy toàn bộ Hướng 2 của Task 1 bằng **một lệnh** (`scripts/run_task1_finetune.py`). Hướng dẫn: `docs/task1_finetune.md`.

1. **Settings:** Accelerator **GPU T4 x2** (dùng 1 GPU), **Internet On** (tài khoản phải xác minh số điện thoại).
2. **Dữ liệu** (repo công khai nên **không ghi ID file Drive vào notebook**), chọn một trong hai:
   - **Add Input** một Kaggle Dataset **private** tạo từ `public-test-cham-diem-phan-hoi-files.zip` và `generate-dataset.zip`
     (Kaggle tự giải nén cũng được: notebook nhận diện gold / public / silver theo nội dung), hoặc
   - **Add-ons → Secrets:** `TASK1_GOLD_PUBLIC_ZIP_ID` và `TASK1_SILVER_ZIP_ID` (ID file Drive), notebook tải bằng `gdown`.
3. Lần đầu để `MODE = 'smoke'` → Run All. Sau đó đổi `MODE = 'full'` → **Save Version → Save & Run All**.

Kết quả (adapter, báo cáo, file nộp) được copy sang `/kaggle/working/task1_qlora`. Dữ liệu sinh viên và checkpoint nằm ở
`/kaggle/tmp`, không lưu cùng version.


In [ ]:
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1'
MODE = 'smoke'   # 'smoke' = 1 bước mỗi giai đoạn + dự đoán 1 bài/đề | 'full' = train đầy đủ + dự đoán 360 bài public
WORK = '/kaggle/tmp'


In [ ]:
%pip install -q "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown


In [ ]:
import os, sys, json, subprocess, hashlib, shutil, zipfile, glob
from datetime import datetime, timezone
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one T4 (16 GB) is enough for 4-bit 7B
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2, rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
RUN_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPO_DIR = Path(WORK) / f'repo-{RUN_TIME}'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
               check=True)
assert (REPO_DIR / 'scripts/run_task1_finetune.py').is_file(), f'Branch {REPO_BRANCH} chưa có code mới: push rồi chạy lại'
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
print('commit', commit)


In [ ]:
# Data: an attached private Kaggle Dataset (ZIPs or folders Kaggle already extracted), else Drive ids from Secrets.
# Each set is recognised by its content, so names and nesting do not matter.
from src.data.loader import resolve_dataset_root

DATA = Path(WORK) / f'task1-data-{RUN_TIME}'
DATA.mkdir(parents=True)


def extract_all(src_dir, dst_dir):
    # Extract every .zip under src_dir (and zips found inside them) into dst_dir, refusing unsafe paths.
    queue = [p for p in Path(src_dir).rglob('*.zip') if '__MACOSX' not in p.parts]
    n = 0
    while queue:
        archive = queue.pop()
        target = dst_dir / f'{n:02d}_{archive.stem}'
        n += 1
        with zipfile.ZipFile(archive) as zf:
            root = target.resolve()
            for info in zf.infolist():
                assert (root / info.filename).resolve().is_relative_to(root), f'unsafe path {info.filename}'
            zf.extractall(target)
        queue += [p for p in target.rglob('*.zip') if '__MACOSX' not in p.parts]


def labelled(d):
    rows = json.loads((d / 'task1_grading.json').read_text(encoding='utf-8'))['samples']
    return bool(rows) and bool(rows[0].get('output'))


def find_sets(*roots):
    sets = {}
    for root in roots:
        for f in sorted(Path(root).rglob('task1_grading.json')):
            d = f.parent
            if '__MACOSX' in d.parts or d.name == 'heldout' or not (d / 'exams.json').exists():
                continue
            if (d / 'heldout' / 'task1_grading.json').exists():
                sets.setdefault('silver', d)
            elif labelled(d):
                sets.setdefault('gold', d)
            else:
                sets.setdefault('public', d)
    return sets


extract_all('/kaggle/input', DATA / 'extracted')
sets = find_sets('/kaggle/input', DATA / 'extracted')
origin = 'kaggle-input'
if len(sets) < 3:  # fall back to Drive ids stored as Kaggle Secrets (never in the public repo)
    from kaggle_secrets import UserSecretsClient
    import gdown
    secrets = UserSecretsClient()
    (DATA / 'drive').mkdir()
    for name, secret_name in [('gold_public.zip', 'TASK1_GOLD_PUBLIC_ZIP_ID'), ('silver.zip', 'TASK1_SILVER_ZIP_ID')]:
        try:
            gdown.download(id=secrets.get_secret(secret_name), output=str(DATA / 'drive' / name), quiet=True)
        except Exception as exc:
            print(f'Secret {secret_name} không dùng được: {type(exc).__name__}')
    extract_all(DATA / 'drive', DATA / 'drive_extracted')
    sets = {**find_sets(DATA / 'drive_extracted'), **sets}
    origin = 'drive (Kaggle Secrets)'
print({k: str(v) for k, v in sets.items()})
missing = {'gold', 'public', 'silver'} - set(sets)
assert not missing, (f'Không tìm thấy {sorted(missing)}. Add Input dataset chứa public-test-cham-diem-phan-hoi-files.zip '
                     'và generate-dataset.zip (Kaggle có thể đã giải nén), hoặc đặt Secrets TASK1_GOLD_PUBLIC_ZIP_ID / TASK1_SILVER_ZIP_ID.')
GOLD_ROOT, PUBLIC_ROOT, SILVER_ROOT = (resolve_dataset_root(sets[k]) for k in ('gold', 'public', 'silver'))


def digest(root):
    h = hashlib.sha256()
    for f in sorted(p for p in Path(root).rglob('*') if p.is_file()):
        h.update(str(f.relative_to(root)).encode())
        h.update(f.read_bytes())
    return h.hexdigest()


manifest = {'prepared_at_utc': RUN_TIME, 'git_commit': commit, 'origin': origin,
            'sets': {k: {'path': str(v), 'sha256': digest(v)} for k, v in
                     {'gold': GOLD_ROOT, 'public': PUBLIC_ROOT, 'silver': SILVER_ROOT}.items()}}
MANIFEST = DATA / 'source_manifest.json'
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest['sets'], indent=1))


In [ ]:
WORK_DIR = Path(WORK) / f'task1-run-{RUN_TIME}'
cmd = [sys.executable, 'scripts/run_task1_finetune.py', '--gold-root', str(GOLD_ROOT), '--public-root', str(PUBLIC_ROOT),
       '--silver-root', str(SILVER_ROOT), '--work-dir', str(WORK_DIR), '--source-manifest', str(MANIFEST)]
# Stream the script's output into this cell (plain subprocess output only reaches the Kaggle console log).
proc = subprocess.Popen(cmd + (['--smoke'] if MODE == 'smoke' else []), cwd=REPO_DIR, text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='')
if proc.wait():
    raise RuntimeError(f'run_task1_finetune.py failed (exit {proc.returncode}): the error is printed just above')


In [ ]:
# Keep only artefacts without student code: adapter, reports, metrics, submission.
RUN_DIR = WORK_DIR / 'qlora' / ('smoke' if MODE == 'smoke' else '')
KEEP = Path('/kaggle/working/task1_qlora') / MODE
KEEP.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / 'best_adapter').exists():
    shutil.copytree(RUN_DIR / 'best_adapter', KEEP / 'best_adapter', dirs_exist_ok=True)
for pattern in ['training_summary.json', 'training_config.json', 'silver_manifest.json', 'source_manifest.json',
                'error_analysis.md', '*/validation_report.json', '*/epoch_*/metrics.json', '*/epoch_*/validation_predictions.json',
                'public_predictions/task1_predictions.json', 'public_predictions/run_info.json']:
    for f in RUN_DIR.glob(pattern):
        dst = KEEP / f.relative_to(RUN_DIR)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dst)
print(chr(10).join(sorted(str(p.relative_to(KEEP)) for p in KEEP.rglob('*') if p.is_file())))
print(open(KEEP / 'error_analysis.md', encoding='utf-8').read() if (KEEP / 'error_analysis.md').exists() else '')
